In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags

df = load_all_snapshots()
f = add_discipline_flags(df)

by_count = (
    f.groupby(["balls", "strikes"])
    .agg(
        pitches=("in_zone", "size"),
        zone_pct=("in_zone", "mean"),
        swing_pct=("is_swing", "mean"),
    )
)
by_count["whiff_pct"] = (
    f[f["is_swing"]].groupby(["balls", "strikes"])["is_whiff"].mean()
)
print(by_count.round(3).sort_index().to_string())

               pitches  zone_pct  swing_pct  whiff_pct
balls strikes                                         
0     0         182540     0.538      0.315      0.246
      1          92740     0.451      0.488      0.259
      2          48768     0.325      0.504      0.257
1     0          68347     0.550      0.428      0.242
      1          71394     0.499      0.543      0.238
      2          70086     0.381      0.575      0.243
2     0          22969     0.579      0.418      0.202
      1          36369     0.559      0.577      0.211
      2          59265     0.468      0.642      0.211
3     0           7312     0.595      0.092      0.127
      1          14960     0.617      0.549      0.179
      2          35882     0.581      0.703      0.172


In [2]:
two_strikes = by_count.xs(2, level="strikes")
print(two_strikes.round(3).to_string())
print()
print("zone% at 0-2:", round(two_strikes.loc[0, "zone_pct"], 3))
print("zone% at 3-2:", round(two_strikes.loc[3, "zone_pct"], 3))
print("difference:  ", round(two_strikes.loc[3, "zone_pct"] - two_strikes.loc[0, "zone_pct"], 3))

       pitches  zone_pct  swing_pct  whiff_pct
balls                                         
0        48768     0.325      0.504      0.257
1        70086     0.381      0.575      0.243
2        59265     0.468      0.642      0.211
3        35882     0.581      0.703      0.172

zone% at 0-2: 0.325
zone% at 3-2: 0.581
difference:   0.256


In [3]:
sw = f[f["is_swing"]]
detail = (
    sw.groupby(["balls", "strikes", "in_zone"])["is_whiff"]
    .agg(["mean", "size"])
    .rename(columns={"mean": "whiff_pct", "size": "swings"})
)
print(detail.xs(2, level="strikes").round(3).to_string())

               whiff_pct  swings
balls in_zone                   
0     False        0.404   10832
      True         0.141   13728
1     False        0.389   16666
      True         0.139   23619
2     False        0.366   13279
      True         0.128   24798
3     False        0.318    6701
      True         0.120   18527


In [4]:
iz = f[f["in_zone"]].copy()
iz["dist_from_center"] = np.hypot(
    pd.to_numeric(iz["plate_x"], errors="coerce"),
    pd.to_numeric(iz["plate_z"], errors="coerce")
    - (pd.to_numeric(iz["sz_top"], errors="coerce")
       + pd.to_numeric(iz["sz_bot"], errors="coerce")) / 2
)

center = iz.groupby(["balls", "strikes"])["dist_from_center"].mean()
print(center.unstack().round(3).to_string())

strikes      0      1      2
balls                       
0        0.643  0.666  0.693
1        0.643  0.654  0.683
2        0.638  0.638   0.66
3        0.623  0.627  0.632
